[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammad-zainal-muttaqin/NulisBuku/blob/main/website/notebooks/ch15.ipynb)

Notebook Bab 15 ini punya dua bagian. Bagian **Demo** tinggal Anda jalankan lalu amati keluarannya; bagian **Mini Project** berisi soal dan data yang Anda kerjakan sendiri.

Karena memakai *pretrained model*, notebook ini paling praktis dijalankan di **Google Colab**. Kita menggabungkan embedding *pretrained* (dipelajari mesin) dengan fitur rancangan tangan menjadi model hibrida.

## Persiapan

In [1]:
%pip install -q sentence-transformers scikit-learn

import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sentence_transformers import SentenceTransformer

print('Setup selesai.')

Setup selesai.


## Section 1 - Demo: Embedding Pretrained + Fitur Rancangan Tangan

## Data dan dua sumber fitur

Fitur rancangan tangan sederhana dari teks (panjang, rasio huruf kapital, kepadatan angka, jumlah tanda seru/tanya) sebagai pelengkap embedding *pretrained*.

In [2]:
cats = ['rec.autos', 'sci.med', 'comp.graphics']
news = fetch_20newsgroups(subset='all', categories=cats,
                          remove=('headers', 'footers', 'quotes'), random_state=42)
texts, y = news.data, news.target

def fitur_tangan(dokumen):
    out = []
    for t in dokumen:
        L = max(len(t), 1)
        kapital = sum(c.isupper() for c in t) / L
        angka = sum(c.isdigit() for c in t) / L
        seru = t.count('!') + t.count('?')
        out.append([len(t), kapital, angka, seru])
    return np.array(out, dtype=float)

eng = fitur_tangan(texts)
print('Dokumen:', len(texts), '| fitur tangan:', eng.shape)

Dokumen: 2953 | fitur tangan: (2953, 4)


In [3]:
enc = SentenceTransformer('all-MiniLM-L6-v2')
emb = enc.encode(texts, batch_size=64, show_progress_bar=False, normalize_embeddings=True)
print('Embedding pretrained:', emb.shape)

Embedding pretrained: (2953, 384)


## Rancangan tangan vs embedding vs hibrida

In [4]:
idx = np.arange(len(y))
itr, ite = train_test_split(idx, test_size=0.25, random_state=42, stratify=y)
sc = StandardScaler().fit(eng[itr])
eng_tr, eng_te = sc.transform(eng[itr]), sc.transform(eng[ite])
emb_tr, emb_te = emb[itr], emb[ite]

def acc(Xtr, Xte):
    clf = LogisticRegression(max_iter=2000).fit(Xtr, y[itr])
    return accuracy_score(y[ite], clf.predict(Xte))

print('Fitur rancangan tangan :', round(acc(eng_tr, eng_te), 3))
print('Embedding pretrained    :', round(acc(emb_tr, emb_te), 3))
print('Hibrida (emb + tangan)  :', round(acc(np.hstack([emb_tr, eng_tr]), np.hstack([emb_te, eng_te])), 3))

Fitur rancangan tangan : 0.493
Embedding pretrained    : 0.938
Hibrida (emb + tangan)  : 0.935


> 🔎 **Amati.** Fitur rancangan tangan sendirian lemah, embedding *pretrained* jauh lebih kuat, dan model hibrida menyamai atau sedikit melampaui embedding. Nilai hibrida terlihat saat fitur rancangan tangan membawa sinyal domain yang tidak tertangkap embedding umum. Pola ini menunjukkan bahwa *deep learning* tidak menghapus rekayasa fitur; ia menggeser perannya menjadi pelengkap.

## Section 2 - Mini Project

## Soal

Bangun model hibrida untuk klasifikasi teks yang menggabungkan embedding *pretrained* dengan fitur rancangan tangan Anda sendiri.

Tugas:

1. Rancang minimal empat fitur rancangan tangan yang relevan (boleh berbeda dari contoh).
2. Bandingkan: rancangan tangan saja, embedding saja, dan hibrida.
3. Analisis kapan fitur rancangan tangan menambah nilai di atas embedding.

**Luaran:** kode fitur + perbandingan akurasi + 3-4 kalimat analisis.

**Kriteria penilaian:** (a) fitur tangan di-*scale* dengan *fit* di train saja; (b) perbandingan adil; (c) analisis peran fitur pelengkap.

In [5]:
# DATA AWAL (jangan diubah)
cats2 = ['rec.sport.baseball', 'sci.electronics', 'talk.religion.misc']
news2 = fetch_20newsgroups(subset='all', categories=cats2,
                           remove=('headers', 'footers', 'quotes'), random_state=7)
texts2, y2 = news2.data, news2.target
print('Dokumen:', len(texts2), '| kelas:', news2.target_names)

Dokumen: 2606 | kelas: ['rec.sport.baseball', 'sci.electronics', 'talk.religion.misc']


In [ ]:
# Kerjakan di sini.
# Petunjuk: buat fungsi fitur tangan Anda; enc.encode(texts2) untuk embedding; np.hstack untuk hibrida.
